# Malicious URL Detection Model Tester

This notebook allows you to test the Random Forest model trained using lexical features as described in the paper:
**"Using Lexical Features for Malicious URL Detection - A Machine Learning Approach"**

In [1]:
import os
import sys

# Add project root to path to allow importing src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Change working directory to project root for model paths to work
os.chdir(project_root)

from src.analysis.lexical.analyzer import LexicalAnalyzer

## 1. Load the LexicalAnalyzer
The analyzer loads the trained Random Forest model, scaler, feature selector, and N-gram extractor.

In [2]:
# Initialize the analyzer (loads all model artifacts)
try:
    analyzer = LexicalAnalyzer()
    print("✅ LexicalAnalyzer loaded successfully.")
    print(f"   - Selected features: {len(analyzer._selected_features)} features")
    print(f"   - Trigram extractor: {'✓' if analyzer._trigram_extractor else '✗'}")
    print(f"   - Scaler: {'✓' if analyzer._scaler else '✗'}")
    print(f"   - SFM Selector: {'✓' if analyzer._selector else '✗'}")
except FileNotFoundError as e:
    print(f"❌ {e}")

✅ LexicalAnalyzer loaded successfully.
   - Selected features: 52 features
   - Trigram extractor: ✓
   - Scaler: ✓
   - SFM Selector: ✓


## 2. Prediction Function
This function uses the LexicalAnalyzer to predict if a URL is malicious.

In [3]:
def predict_url(url: str):
    """Analyzes a URL and displays the prediction."""
    try:
        result = analyzer.analyze(url)
        
        # Colors for visualization
        color = "\033[91m" if result["is_malicious"] else "\033[92m"
        reset = "\033[0m"
        
        print(f"URL: {url}")
        print(f"Result: {color}{result['prediction']}{reset} (Confidence: {result['score']:.2%})")
        print(f"  Key Features:")
        for k, v in result["features"].items():
            print(f"    - {k}: {v}")
        print("-" * 50)
        
        return result
    except Exception as e:
        print(f"Error processing {url}: {e}")
        return None

# Test some samples
print("Running test samples...\n")
predict_url("https://google.com")
predict_url("secure-login-account.top/verify")
predict_url("http://123.45.67.89/malware.exe")
predict_url("wikipedia.org")

Running test samples...

URL: https://google.com
Result: Benign (Confidence: 7.11%)
  Key Features:
    - url_length: 18.0
    - url_entropy: 3.5724312513221195
    - domain_length: 10.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 0.0
    - subdomain_levels: 1.0
--------------------------------------------------
URL: secure-login-account.top/verify
Result: Malicious (Confidence: 97.00%)
  Key Features:
    - url_length: 31.0
    - url_entropy: 4.106949132758152
    - domain_length: 24.0
    - has_sensitive_keyword: 1.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 1.0
    - subdomain_levels: 1.0
--------------------------------------------------


d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


URL: http://123.45.67.89/malware.exe
Result: Malicious (Confidence: 100.00%)
  Key Features:
    - url_length: 31.0
    - url_entropy: 4.260332600569877
    - domain_length: 12.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 1.0
    - suspicious_tld: 0.0
    - subdomain_levels: 3.0
--------------------------------------------------
URL: wikipedia.org
Result: Benign (Confidence: 0.00%)
  Key Features:
    - url_length: 13.0
    - url_entropy: 3.334679141051595
    - domain_length: 13.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 0.0
    - subdomain_levels: 1.0
--------------------------------------------------


{'score': 9.611687812379855e-07,
 'is_malicious': False,
 'prediction': 'Benign',
 'features': {'url_length': 13.0,
  'url_entropy': 3.334679141051595,
  'domain_length': 13.0,
  'has_sensitive_keyword': 0.0,
  'has_suspicious_extension': 0.0,
  'suspicious_tld': 0.0,
  'subdomain_levels': 1.0}}

## 3. Test Your Own URL
Enter a URL below to test it against the model.

In [ ]:
user_url = "https://youtube.com/"  # @param {type:"string"}
if user_url:
    predict_url(user_url)
else:
    print("Enter a URL in the field above (or edit the code) to test it.")

URL: https://www.youtube.com/
Result: Malicious (Confidence: 98.00%)
  Key Features:
    - url_length: 24.0
    - url_entropy: 3.7406015629507228
    - domain_length: 15.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 0.0
    - subdomain_levels: 2.0
--------------------------------------------------


d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


## 4. Batch Testing
Test multiple URLs at once.

In [5]:
test_urls = [
    "https://github.com",
    "https://stackoverflow.com",
    "http://suspicious-login.xyz/update-account",
    "http://192.168.1.1/admin/config.php",
    "https://amazon.com",
    "free-prize-winner.top/claim",
]

print("Batch analysis results:\n")
for url in test_urls:
    predict_url(url)

Batch analysis results:

URL: https://github.com
Result: Benign (Confidence: 11.00%)
  Key Features:
    - url_length: 18.0
    - url_entropy: 3.6835423624332306
    - domain_length: 10.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 0.0
    - subdomain_levels: 1.0
--------------------------------------------------
URL: https://stackoverflow.com
Result: Malicious (Confidence: 57.00%)
  Key Features:
    - url_length: 25.0
    - url_entropy: 4.023465189601646
    - domain_length: 17.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 0.0
    - subdomain_levels: 1.0
--------------------------------------------------
URL: http://suspicious-login.xyz/update-account
Result: Malicious (Confidence: 100.00%)
  Key Features:
    - url_length: 42.0
    - url_entropy: 4.189238255803027
    - domain_length: 20.0
    - has_sensitive_keyword: 1.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 1.0
    - s

d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


URL: http://192.168.1.1/admin/config.php
Result: Malicious (Confidence: 100.00%)
  Key Features:
    - url_length: 35.0
    - url_entropy: 4.0791433740260095
    - domain_length: 11.0
    - has_sensitive_keyword: 1.0
    - has_suspicious_extension: 1.0
    - suspicious_tld: 0.0
    - subdomain_levels: 3.0
--------------------------------------------------
URL: https://amazon.com
Result: Benign (Confidence: 10.11%)
  Key Features:
    - url_length: 18.0
    - url_entropy: 3.6143694458867563
    - domain_length: 10.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 0.0
    - subdomain_levels: 1.0
--------------------------------------------------
URL: free-prize-winner.top/claim
Result: Malicious (Confidence: 97.00%)
  Key Features:
    - url_length: 27.0
    - url_entropy: 3.8841550945958057
    - domain_length: 21.0
    - has_sensitive_keyword: 0.0
    - has_suspicious_extension: 0.0
    - suspicious_tld: 1.0
    - subdomain_levels: 1.0
-------

d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\Albert\GitRepo\Agent-Vanguard\CyberSentinel\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
